# HybridEnhancementNet
Low-light image enhancement for exploration rover deployment.

**Pipeline:**
1. Environment setup (Drive + GitHub + packages)
2. Dataset download (LOL + DarkFace via Kaggle)
3. LOL supervised training
4. DarkFace self-supervised training
5. Evaluation + export
6. Push results to GitHub

## 1. Environment Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/HybridEnhancementNet.git'
REPO_DIR = '/content/HybridEnhancementNet'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
!pip install -r requirements.txt -q

In [ ]:
import sys
sys.path.append('/content/HybridEnhancementNet')

import yaml, torch
from src.utils import load_config, setup_kaggle_credentials, download_datasets, make_drive_dirs, get_device

cfg    = load_config('config.yaml')
device = get_device()

setup_kaggle_credentials(cfg['paths']['kaggle_json'])
make_drive_dirs(cfg)

print('Environment ready.')

## 2. Download Datasets

In [ ]:
from src.utils import download_datasets
lol_path, darkface_path = download_datasets(cfg)
print(f'LOL path     : {lol_path}')
print(f'DarkFace path: {darkface_path}')

## 3. LOL Supervised Training

In [ ]:
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from src.dataset import LOLDataset
from src.model import HybridEnhancementNet
from src.evaluate import visualize_dataset_samples

lol_cfg   = cfg['training']['lol']
transform = transforms.Compose([transforms.ToTensor()])

full_dataset = LOLDataset(lol_path, transform=transform, train=True)
visualize_dataset_samples(full_dataset,
                          save_path=cfg['paths']['results'] + '/lol_samples.png',
                          title='LOL dataset samples',
                          paired=True)

train_size = int(lol_cfg['train_split'] * len(full_dataset))
val_size   = len(full_dataset) - train_size
train_set, val_set = random_split(full_dataset, [train_size, val_size],
                                   generator=torch.Generator().manual_seed(lol_cfg['seed']))

train_loader = DataLoader(train_set, batch_size=lol_cfg['batch_size'], shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_set,   batch_size=lol_cfg['batch_size'], shuffle=False, num_workers=2)

print(f'Train: {len(train_set)} | Val: {len(val_set)}')

In [ ]:
from src.train import train_lol, save_metrics_csv, print_metrics_table
from src.evaluate import compute_psnr_mse, visualize_enhancement, plot_training_curves

lol_model = HybridEnhancementNet().to(device)

train_losses, val_losses, lrs = train_lol(
    lol_model, train_loader, val_loader, device,
    cfg=lol_cfg,
    save_dir=cfg['paths']['checkpoints']
)

plot_training_curves(
    train_losses, val_losses, lrs,
    save_path=cfg['paths']['results'] + '/lol_training_history.png',
    title='LOL Training History'
)

mse, psnr = compute_psnr_mse(lol_model, val_loader, device, paired=True)
visualize_enhancement(
    lol_model, val_loader, device,
    save_path=cfg['paths']['results'] + '/lol_results.png',
    tag='LOL final',
    paired=True
)

## 4. DarkFace Self-Supervised Training

In [ ]:
from torch.utils.data import random_split
from src.dataset import DarkFaceDataset, darkface_collate_fn
from src.utils import preflight_check

df_cfg = cfg['training']['darkface']
preflight_check(darkface_path)

df_dataset = DarkFaceDataset(darkface_path, transform=transform)

train_sz = int(df_cfg['train_split'] * len(df_dataset))
test_sz  = len(df_dataset) - train_sz
df_train, df_test = random_split(df_dataset, [train_sz, test_sz],
                                  generator=torch.Generator().manual_seed(df_cfg['seed']))

df_train_loader = DataLoader(df_train, batch_size=df_cfg['batch_size'], shuffle=True,
                              collate_fn=darkface_collate_fn)
df_test_loader  = DataLoader(df_test,  batch_size=df_cfg['batch_size'], shuffle=False,
                              collate_fn=darkface_collate_fn)

print(f'DarkFace Train: {train_sz} | Test: {test_sz}')

In [ ]:
from src.train import train_darkface, print_metrics_table, save_metrics_csv

df_model    = HybridEnhancementNet().to(device)
metrics_log = []

train_losses, lrs = train_darkface(
    df_model, df_train_loader, df_test_loader, device,
    cfg=df_cfg,
    save_dir=cfg['paths']['checkpoints'],
    vis_interval=df_cfg['vis_interval'],
    metrics_log=metrics_log
)

plot_training_curves(
    train_losses, [], lrs,
    save_path=cfg['paths']['results'] + '/darkface_training_history.png',
    title='DarkFace Training History'
)

print_metrics_table(metrics_log)
save_metrics_csv(metrics_log, cfg['paths']['results'] + '/metrics.csv')

## 5. Export Model for Rover

In [ ]:
from deploy.export_onnx import export_onnx

export_onnx(
    checkpoint_path=cfg['paths']['checkpoints'] + '/best_model.pth',
    output_path=cfg['paths']['drive_base'] + '/exported/model.onnx',
    input_h=cfg['deploy']['input_size'][1],
    input_w=cfg['deploy']['input_size'][2]
)

## 6. Push to GitHub

In [ ]:
!cd /content/HybridEnhancementNet && \
  git config user.email 'your@email.com' && \
  git config user.name 'Your Name' && \
  git add notebooks/ src/ config.yaml deploy/ && \
  git commit -m 'training run: update model and results' && \
  git push